# Lunar terrain and native environmental crops

**Output meaning:** this stage deliberately masks finite source heights using image support. Its elevation holes are processing-policy exclusions, not absent source elevation. [Notebook 13](13_build_lunar_terrain_views.ipynb) creates the full-context and image-supported views used by [current terrain EDA](../exploration/14_terrain_feature_eda.ipynb).


Create 30 m physical terrain descriptors from the seven 5 m-grid SDEMs, retaining image-count support and quality layers. Crop LOLA and illumination maps at their own native pixel spacing for each SDEM footprint. This notebook deliberately does not upscale environmental maps to 30 m or calculate a combined suitability score.

All lunar grids use the spherical Moon, radius 1,737,400 m; Earth EPSG:4326 is never used for lunar coordinates. SDEM heights are meters relative to that radius. The 5 m SfS grid is not a guarantee of 5 m independent measurements. Average SDEM elevation using only cells with image support; preserve support fractions and quality layers separately.


Run cells in order manually. Raw inputs are read-only. Each processing run writes to a new directory under `data/processed/`; only a `provenance.json` with `status: complete` indicates successful completion. See the [notebook index](../README.md) for environment setup and dependencies.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "outline.md").is_file() and (p / "data/raw").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project root or a folder under notebooks")
sys.path.insert(0, str(ROOT / "notebooks"))
print("Project:", ROOT)


In [ ]:
import json
import numpy as np
import rasterio
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.warp import reproject
from processing import Run, MOON_CRS, Grid, warp, make_grid, roi_mask, write_raster, terrain, crop_native

SPACING_M = 30
ROUGHNESS_WINDOW = 11
MIN_IMAGE_SUPPORT_FRACTION = 0.99
sdem_root = ROOT / "data/raw/moon/sdem/regions"
REGION_NAMES = sorted(p.name for p in sdem_root.iterdir() if p.is_dir())
# Set REGION_NAMES to one exact folder name for a first run.
regions = {}
for name in REGION_NAMES:
    elevation_paths = list((sdem_root / name).glob("*_GLDELEV_005.tif"))
    mask_paths = list((sdem_root / name).glob("*_GLDMASK_005.tif"))
    if len(elevation_paths) != 1 or len(mask_paths) != 1:
        raise ValueError(f"Missing/ambiguous elevation or image-count mask: {name}")
    with rasterio.open(elevation_paths[0]) as src:
        assert src.crs == CRS.from_string(MOON_CRS)
        regions[name] = {"elevation": elevation_paths[0], "mask": mask_paths[0], "bounds": tuple(src.bounds)}
    print(name, regions[name]["bounds"])
lola_root = ROOT / "data/raw/moon/lola_product90/large_area_dem"
lola_names = ["LDEM_80S_40MPP_ADJ.TIF", "LDEM_80S_40MPP_ADJ_EFFRES.TIF", "LDEM_80S_40MPP_ADJ_ERR.TIF",
              "LDEC_80S_40MPP_ADJ.TIF", "LDSM_80S_40MPP_ADJ.TIF", "LDSM_80S_40MPP_ADJ_ERR.TIF"]
illumination = sorted(p for p in (ROOT / "data/raw/moon/illumination_2016").rglob("*") if p.suffix.lower() == ".tif")
assert len(illumination) == 10 and all((lola_root / n).is_file() for n in lola_names)
assert SPACING_M >= 30 and 0 < MIN_IMAGE_SUPPORT_FRACTION <= 1


In [ ]:
config = {"regions": REGION_NAMES, "spacing_m": SPACING_M, "roughness_window": ROUGHNESS_WINDOW,
          "minimum_image_support_fraction": MIN_IMAGE_SUPPORT_FRACTION, "moon_radius_m": 1737400,
          "image_count_alignment": "nearest to elevation grid; missing coverage unsupported",
          "elevation_aggregation": "average supported source cells", "environment": "native grid crops, scaled values"}
with Run(ROOT, "lunar_terrain_environment", config, "preprocessing/05_lunar_terrain_environment.ipynb") as run:
    summaries = {}
    for name, info in regions.items():
        run.input(info["elevation"])
        run.input(info["mask"])
        run.input(sdem_root / name / "README.txt")
        grid = make_grid(info["bounds"], MOON_CRS, SPACING_M)
        with rasterio.open(info["elevation"]) as src:
            heights = src.read(1, masked=True).astype("float32").filled(np.nan)
            heights = heights * src.scales[0] + src.offsets[0]
            # Count masks may be 15 m or have smaller extents than the 5 m elevation.
            source_grid = Grid(src.crs, src.transform, src.width, src.height)
            counts = warp(info["mask"], source_grid, resampling=Resampling.nearest)
            supported = np.isfinite(heights) & np.isfinite(counts) & (counts > 0)
            heights[~supported] = np.nan
            elevation = np.full((grid.height, grid.width), np.nan, dtype="float32")
            fraction = np.full_like(elevation, np.nan)
            for source, destination in ((heights, elevation), (supported.astype("float32"), fraction)):
                reproject(source, destination, src_transform=src.transform, src_crs=src.crs,
                          src_nodata=np.nan, dst_transform=grid.transform, dst_crs=grid.crs,
                          dst_nodata=np.nan, resampling=Resampling.average)
        del heights, counts, supported
        inside = roi_mask(grid, info["bounds"], MOON_CRS)
        elevation[~inside] = np.nan
        fraction[~inside] = np.nan
        elevation[fraction < MIN_IMAGE_SUPPORT_FRACTION] = np.nan
        folder = f"moon/{name}/grid_{SPACING_M}m"
        write_raster(run.path(f"{folder}/elevation_m.tif"), elevation, grid, "m", vertical_reference="radius 1737400 m")
        write_raster(run.path(f"{folder}/image_support_fraction.tif"), fraction, grid, "fraction")
        for feature, values in terrain(elevation, SPACING_M, ROUGHNESS_WINDOW).items():
            units = "degree" if feature == "slope_deg" else "1/m" if feature == "laplacian_per_m" else "m"
            write_raster(run.path(f"{folder}/{feature}.tif"), values, grid, units)
        native = []
        # Preserve source-defined image counts, uncertainty, best resolution and supplied terrain metrics.
        quality = [p for p in (sdem_root / name).rglob("*.tif")
                   if any(token in p.name for token in ("GLDMASK", "GLDSIGM", "GLDISGM", "GLDBRES", "GLDSBCT", "GLDTSLP", "GLDTRGH"))]
        for source in quality + [lola_root / n for n in lola_names] + illumination:
            run.input(source)
            category = "sdem_quality" if source.is_relative_to(sdem_root) else "lola" if source.parent == lola_root else "illumination"
            output = run.path(f"moon/{name}/native/{category}/{source.name}")
            # The region README documents the CRS omitted from slope/roughness GeoTIFFs.
            documented_sdem = category == "sdem_quality" and source.parent.name in ("SLOPE_MAPS", "ROUGHNESS_MAPS")
            metadata = crop_native(source, output, info["bounds"], MOON_CRS,
                                   fallback_crs=MOON_CRS if documented_sdem else None,
                                   crs_evidence=str(sdem_root / name / "README.txt") if documented_sdem else None)
            native.append(metadata)
        summaries[name] = {"bounds_m": info["bounds"], "valid_elevation_fraction": float(np.isfinite(elevation).mean()),
                           "native_layers": native}
        run.path(f"moon/{name}/quality.json").write_text(json.dumps(summaries[name], indent=2), encoding="utf-8")
        print(name, "valid elevation fraction", summaries[name]["valid_elevation_fraction"])
    run.record["summaries"] = summaries
print("Completed:", run.directory)


## Environmental interpretation
The illumination TIFFs contain packed values. Crops mask nodata first, then apply each source's scale and offset exactly once; output TIFFs store decoded values with identity scaling. Source tags and packing values are retained in `quality.json`. Units missing from GeoTIFF metadata remain `source_units_unconfirmed` until checked against product documentation. In particular, do not assume sky visibility uses the same units as illumination. No continuous-darkness duration is derived from mean maps. Native LOLA layers provide context/uncertainty, not additional independent high-resolution SDEM evidence.
